In [1]:
import pandas as pd

interactions = pd.read_csv("data_original/interactions.csv")
users = pd.read_csv("data_original/users.csv")
items = pd.read_csv("data_original/items.csv")

interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])

In [2]:
start = pd.Timestamp("2021-08-02")
end = start + pd.Timedelta(days=7)

history = interactions[interactions["last_watch_dt"] < start]
future = interactions[
    (interactions["last_watch_dt"] >= start) &
    (interactions["last_watch_dt"] < end)
]

print(len(history), len(future))

4266013 383149


In [3]:
popular_items = (
    history.groupby("item_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .to_numpy()
)

popular_items[:10]

array([10440, 15297, 13865,  9728,  4151,  2657,  3734,  4880,   142,
        6809])

In [4]:
seen_items = history.groupby("user_id")["item_id"].agg(set)
warm_future = future[future["user_id"].isin(seen_items.index)]

candidates = {}

for user_id in warm_future["user_id"].unique():
    seen = seen_items[user_id]
    candidates[user_id] = [
        item_id for item_id in popular_items
        if item_id not in seen
    ][:500]

In [5]:
future_items = warm_future.groupby("user_id")["item_id"].agg(set)

recall = {}

for k in [50, 100, 200, 500]:
    scores = []

    for user_id, targets in future_items.items():
        targets = targets - seen_items[user_id]

        if not targets:
            continue

        predicted = set(candidates[user_id][:k])
        scores.append(len(predicted & targets) / len(targets))

    recall[k] = sum(scores) / len(scores)

pd.Series(recall, name="Recall")

50     0.301158
100    0.378346
200    0.487848
500    0.630408
Name: Recall, dtype: float64

In [6]:
import numpy as np
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import BM25Recommender

user_codes, user_index = pd.factorize(history["user_id"])
item_codes, item_index = pd.factorize(history["item_id"])

user_item = csr_matrix(
    (np.ones(len(history), dtype=np.float32), (user_codes, item_codes)),
    shape=(len(user_index), len(item_index))
)


/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
bm25 = BM25Recommender(K=200)
bm25.fit(user_item)

/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.03912997245788574 seconds
  warnings.warn(
100%|██████████| 15237/15237 [00:00<00:00, 43345.30it/s]


In [8]:
user_to_idx = pd.Series(np.arange(len(user_index)), index=user_index)

eval_users = warm_future["user_id"].unique()
eval_indices = user_to_idx.loc[eval_users].to_numpy()

In [9]:
bm25_candidates = {}

batch_size = 1024

for offset in range(0, len(eval_users), batch_size):
    batch_users = eval_users[offset:offset + batch_size]
    batch_indices = eval_indices[offset:offset + batch_size]

    ids, scores = bm25.recommend(
        batch_indices,
        user_item[batch_indices],
        N=500,
        filter_already_liked_items=True
    )

    for user_id, item_ids in zip(batch_users, ids):
        valid = item_ids >= 0
        bm25_candidates[user_id] = item_index[item_ids[valid]].tolist()

In [10]:
bm25_recall = {}

for k in [50, 100, 200, 500]:
    scores = []

    for user_id, targets in future_items.items():
        targets = targets - seen_items[user_id]

        if not targets:
            continue

        predicted = set(bm25_candidates[user_id][:k])
        scores.append(len(predicted & targets) / len(targets))

    bm25_recall[k] = np.mean(scores)

pd.DataFrame({
    "Popularity": recall,
    "BM25": bm25_recall
})

,Popularity,BM25
50,0.301158,0.164792
100,0.378346,0.242882
200,0.487848,0.340867
500,0.630408,0.503252


In [11]:
rrf_candidates = {}

for user_id in eval_users:
    scores = {}

    for rank, item_id in enumerate(candidates[user_id], start=1):
        scores[item_id] = 1 / (60 + rank)

    for rank, item_id in enumerate(bm25_candidates[user_id], start=1):
        scores[item_id] = scores.get(item_id, 0) + 1 / (60 + rank)

    rrf_candidates[user_id] = sorted(
        scores,
        key=scores.get,
        reverse=True
    )[:500]

In [12]:
rrf_recall = {}

for k in [50, 100, 200, 500]:
    scores = []

    for user_id, targets in future_items.items():
        targets = targets - seen_items[user_id]

        if not targets:
            continue

        predicted = set(rrf_candidates[user_id][:k])
        scores.append(len(predicted & targets) / len(targets))

    rrf_recall[k] = np.mean(scores)

pd.DataFrame({
    "Popularity": recall,
    "BM25": bm25_recall,
    "RRF": rrf_recall
})

,Popularity,BM25,RRF
50,0.301158,0.164792,0.357903
100,0.378346,0.242882,0.445648
200,0.487848,0.340867,0.556282
500,0.630408,0.503252,0.704936


In [13]:
oracle_scores = []

for user_id, targets in future_items.items():
    targets = targets - seen_items[user_id]

    if not targets:
        continue

    pool = set(candidates[user_id]) | set(bm25_candidates[user_id])
    oracle_scores.append(len(pool & targets) / len(targets))

np.mean(oracle_scores)

np.float64(0.772829504601174)

Я изначально взял за основе Popularity как самый просто из подходов, потому что если смотреть на статистику KION, то top-1% фильмов и сериалов собирают около 45% всех взаимодействий. BM25 я решил добавить, чтобы генератор кандидатов мог учитывать не только популярность самих фильмов глобально, но и релевантность этих фильмов для конкретных пользователей.